# Activity: How Sure Is the ETA? — FreshKart Delivery-Time Model

**Unit 4 · AI-Assisted Visualization · Visualizing ML Predictions, Probability and Uncertainty**
Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## The case

FreshKart's app shows every customer an **ETA** ("arrives in 24 min") and a range ("**90% range: 22–26 min**"). A second model sorts each order into a speed band, **Fast** (≤ 20 min), **On-time** (21–30 min) or **Late** (> 30 min), and FreshKart plans to send a *"your order may be late"* SMS when the model is confident an order will be late.

The model was trained on **Bengaluru and Pune** orders of **up to 8 km**. Hyderabad launched recently.

Product manager **Anjali** wants to put the following claims on a slide for the leadership review. You have one month of predictions with the real outcomes. **Check every claim with the data and the right visualization.**

| # | Claim |
|---|---|
| 1 | "Our ETA is accurate: the **average error is under 1 minute**." |
| 2 | "The app's **90% range** contains the real delivery time **9 times out of 10**." |
| 3 | "When the model says an order is **90% likely to be late**, it is late about 90% of the time." |
| 4 | "**Temperature scaling** will make the speed-band model **more accurate**." |
| 5 | "The model works **equally well in all three cities**." |
| 6 | "The speed-band model beats the simple rule *always say On-time*." |
| 7 | "For long Hyderabad orders the model is **just as sure** as anywhere else, so we can trust its ETA there." |

**Ground rules (from the lecture):** show more than the top-1 label · a confidence band is not a prediction interval · confidence is not probability until you check calibration on held-out data · say what every interval means · write uncertainty in plain words for customers.

**How to work:** run the notebook top to bottom. Complete every cell marked `# TODO` (replace `None` with your code or answer) and write in the cells marked **✏️ Your answer**. Cells that depend on your TODO will tell you if something is still missing.

### The data: `freshkart_eta_predictions.csv` (synthetic)

| Column | Meaning |
|---|---|
| `order_id`, `order_date`, `city`, `zone` | the order |
| `distance_km`, `items` | inputs to the model |
| `actual_minutes` | real delivery time |
| `pred_minutes`, `pred_lo_90`, `pred_hi_90` | the ETA shown in the app and its "90% range" |
| `p_fast`, `p_ontime`, `p_late` | speed-band probabilities from the classifier (sum to 1) |
| `actual_band` | the real band: Fast / On-time / Late |
| `ensemble_sd` | spread (SD, minutes) of the ETA across 5 retrained copies of the model |
| `split` | `calib` = held-out orders you may use to *fit* corrections; `test` = orders you use to *evaluate* them |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.special import softmax, logsumexp

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (6, 3.2), "axes.spines.top": False, "axes.spines.right": False})
GREY, ACCENT, BLUE = "#bdbdbd", "#d95f02", "#4c78a8"
BANDS = ["Fast", "On-time", "Late"]
PCOLS = ["p_fast", "p_ontime", "p_late"]

def ready(*vals):
    """Helper: tells you if a TODO above is still unanswered."""
    ok = all(v is not None for v in vals)
    if not ok:
        print("⚠ Complete the TODO cell(s) above first, then re-run this cell.")
    return ok

df = pd.read_csv("freshkart_eta_predictions.csv", parse_dates=["order_date"])
df["error"] = df["actual_minutes"] - df["pred_minutes"]          # positive = late versus the ETA
calib, test = df[df.split == "calib"].copy(), df[df.split == "test"].copy()
df.head()

---
## Part 1 · Predicted versus actual (claim 1)

In [ ]:
mean_error = None   # TODO: mean of df["error"]
mae = None          # TODO: mean absolute error

if ready(mean_error, mae):
    print(f"mean error = {mean_error:+.2f} min | mean absolute error = {mae:.2f} min")
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(df.actual_minutes, df.pred_minutes, s=4, alpha=.25, color=BLUE)
    lim = [5, 75]; ax.plot(lim, lim, "k--", lw=1, label="perfect prediction")
    binned = df.groupby(pd.cut(df.actual_minutes, np.arange(5, 80, 5)), observed=True).agg(a=("actual_minutes", "mean"), p=("pred_minutes", "mean"))
    ax.plot(binned.a, binned.p, color=ACCENT, marker="o", lw=2, label="average prediction per 5-min bin")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal"); ax.set_xlabel("actual minutes"); ax.set_ylabel("predicted minutes"); ax.legend()
    plt.show()

In [ ]:
bias = df.groupby(pd.cut(df.distance_km, [0, 2, 4, 6, 8, 14]), observed=True)["error"].agg(["mean", "size"]).round(2)
display(bias)

short_bias = None   # TODO: mean error for orders up to 2 km (read it from the table or compute it)
long_bias = None    # TODO: mean error for orders over 8 km

if ready(short_bias, long_bias):
    print(f"<= 2 km: {short_bias:+.1f} min | > 8 km: {long_bias:+.1f} min")

**✏️ Your answer (Part 1):** Claim 1 quotes the *mean* error. What does the predicted-vs-actual chart show that the mean hides, and how would you rewrite claim 1?

*Your answer:*

---
## Part 2 · What does the "90% range" really cover? (claim 2)

In [ ]:
inside = None   # TODO: boolean Series, True when actual_minutes lies between pred_lo_90 and pred_hi_90 (inclusive)

if ready(inside):
    print(f"Share of orders inside the app's '90% range': {inside.mean():.1%}")
    print(f"Typical width of the range: {(df.pred_hi_90 - df.pred_lo_90).median():.1f} min | typical |error|: {df.error.abs().median():.1f} min")

The app's range was built from the **standard error of the model's average prediction**: it is a *confidence band for the mean*, not a *prediction interval* for one order. Build a proper 90% prediction interval: use the **calibration** orders to estimate how far individual orders fall from the ETA, then **evaluate on the test** orders.

In [ ]:
resid_sd = None   # TODO: standard deviation of the error on the CALIB orders
half_90 = None    # TODO: half-width of a 90% normal interval: 1.645 * resid_sd

# (alternative without a normal assumption: the 90% quantile of |error| on calib orders, a split-conformal interval)
half_conformal = np.quantile(calib["error"].abs(), 0.90)

if ready(resid_sd, half_90):
    test["pi_lo"], test["pi_hi"] = test.pred_minutes - half_90, test.pred_minutes + half_90
    test["in_pi"] = test.actual_minutes.between(test.pi_lo, test.pi_hi)
    print(f"normal PI: ETA ± {half_90:.1f} min -> test coverage {test.in_pi.mean():.1%}")
    print(f"conformal PI: ETA ± {half_conformal:.1f} min -> test coverage {(test.error.abs() <= half_conformal).mean():.1%}")
    display(test.groupby("city")["in_pi"].mean().round(3).rename("coverage of the new 90% PI"))

    s = test.sample(40, random_state=4).sort_values("pred_minutes").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.errorbar(s.index, s.pred_minutes, yerr=[s.pred_minutes - s.pi_lo, s.pi_hi - s.pred_minutes], fmt="none", color=BLUE, alpha=.35, lw=6, label="new 90% prediction interval")
    ax.errorbar(s.index, s.pred_minutes, yerr=[s.pred_minutes - s.pred_lo_90, s.pred_hi_90 - s.pred_minutes], fmt="none", color="k", lw=1.5, capsize=3, label="app's '90% range'")
    ax.scatter(s.index, s.actual_minutes, color=ACCENT, s=18, zorder=3, label="actual")
    ax.set_xticks([]); ax.set_ylabel("minutes"); ax.legend(ncol=3, loc="upper left"); ax.set_title("40 test orders, sorted by ETA", loc="left"); plt.show()

**✏️ Your answer (Part 2):** (a) Explain to Anjali, in two sentences, the difference between a confidence band and a prediction interval. (b) Which city does the new interval under-cover, and what does that tell you about using **one** residual SD for every order?

*Your answer:*

---
## Part 3 · Class probabilities, margins and prediction sets

In [ ]:
P = df[PCOLS].values
df["pred_band"] = np.array(BANDS)[P.argmax(1)]
df["confidence"] = P.max(axis=1)
srt = np.sort(P, axis=1)

df["margin"] = None     # TODO: top probability minus second probability (hint: srt[:, -1] - srt[:, -2])

if ready(*df["margin"].head(1)):
    near_tie_share = (df["margin"].astype(float) < 0.1).mean()
    print(f"Orders whose top two bands are within 0.10 of each other: {near_tie_share:.1%}")
    s = df.sample(30, random_state=1).sort_values("margin")
    fig, ax = plt.subplots(figsize=(5, 6.5))
    ax.imshow(s[PCOLS].values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for r in range(len(s)):
        for c in range(3): ax.text(c, r, f"{s[PCOLS].values[r, c]:.2f}", ha="center", va="center", fontsize=7)
        ax.text(3.0, r, f"actual {s.actual_band.iloc[r]}", va="center", fontsize=7, color="k" if s.actual_band.iloc[r] == s.pred_band.iloc[r] else ACCENT)
    ax.set_xticks(range(3)); ax.set_xticklabels(BANDS); ax.set_yticks([]); ax.set_xlim(-.5, 4)
    ax.set_title("30 orders sorted by margin", loc="left"); plt.show()

A **prediction set** lists every band the model cannot rule out. A simple rule: add bands from most to least likely until their probabilities add up to at least 0.9.

In [ ]:
def prediction_set(p_row, level=0.9):
    order = np.argsort(p_row)[::-1]
    chosen, total = [], 0.0
    for k in order:
        # TODO: add BANDS[k] to chosen and its probability to total; stop once total >= level
        pass
    return chosen

example = df.iloc[int(df["margin"].astype(float).idxmin())] if df["margin"].notna().all() else df.iloc[0]
print("probabilities:", {b: round(float(v), 3) for b, v in zip(BANDS, example[PCOLS])}, "->", prediction_set(example[PCOLS].values.astype(float)))
if len(prediction_set(example[PCOLS].values.astype(float))) > 0:
    df["set_size"] = [len(prediction_set(r)) for r in P]
    print(df["set_size"].value_counts().sort_index().rename("orders with this many bands in the set"))

**✏️ Your answer (Part 3):** For a customer, how would you show an order whose set is {On-time, Late}, instead of the single label "On-time"?

*Your answer:*

---
## Part 4 · Is "90% likely to be late" really 90%? (claims 3 and 4)

### 4a · Reliability diagram for the "Late" probability (test orders)

In [ ]:
def reliability_table(prob, outcome, n_bins=10):
    bins = np.clip((prob * n_bins).astype(int), 0, n_bins - 1)
    t = pd.DataFrame({"bin": bins, "p": prob, "y": outcome}).groupby("bin").agg(mean_p=("p", "mean"), freq=("y", "mean"), n=("y", "size"))
    t["se"] = np.sqrt(t.freq * (1 - t.freq) / t.n)
    return t

y_late = (test.actual_band == "Late").astype(int).values
rel = reliability_table(test.p_late.values, y_late)

ece_late = None   # TODO: expected calibration error = sum over bins of (n / total n) * |freq - mean_p|
brier_model = None      # TODO: mean of (p_late - y_late)^2 on test orders
brier_baseline = None   # TODO: Brier score of always predicting the share of late orders in the CALIB set

if ready(ece_late, brier_model, brier_baseline):
    print(f"ECE = {ece_late:.3f} | Brier: model {brier_model:.3f} vs base-rate baseline {brier_baseline:.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].errorbar(rel.mean_p, rel.freq, yerr=1.96 * rel.se, fmt="o-", color=BLUE, capsize=3)
    ax[0].set_xlabel("predicted P(late)"); ax[0].set_ylabel("actual share late"); ax[0].set_title("Reliability diagram (test)")
    ax[1].bar(rel.mean_p, rel.n, width=.07, color=GREY); ax[1].set_xlabel("predicted P(late)"); ax[1].set_title("orders per bin")
    plt.tight_layout(); plt.show()
    top = test.p_late >= 0.85
    print(f"Orders with P(late) >= 0.85: {top.sum()}, of which actually late: {y_late[top.values].mean():.0%}")

### 4b · Temperature scaling, fitted on the calibration orders

In [ ]:
def nll(T, logp, y):
    z = logp / T
    return float(-(z - logsumexp(z, axis=1, keepdims=True))[np.arange(len(y)), y].mean())

band_index = {b: i for i, b in enumerate(BANDS)}
logp_cal, y_cal = np.log(calib[PCOLS].values), calib.actual_band.map(band_index).values
logp_te, y_te = np.log(test[PCOLS].values), test.actual_band.map(band_index).values

T = None          # TODO: the T that minimises nll on the CALIB orders (hint: minimize_scalar(lambda t: nll(t, logp_cal, y_cal), bounds=(0.5, 20), method="bounded").x)

if ready(T):
    P_T = softmax(logp_te / T, axis=1)
    labels_same = (P_T.argmax(1) == logp_te.argmax(1)).all()
    acc_before, acc_after = (logp_te.argmax(1) == y_te).mean(), (P_T.argmax(1) == y_te).mean()
    rel_T = reliability_table(P_T[:, 2], y_late)
    ece_T = float((rel_T.n / rel_T.n.sum() * (rel_T.freq - rel_T.mean_p).abs()).sum())
    print(f"T = {T:.2f} | labels unchanged: {labels_same} | accuracy {acc_before:.3f} -> {acc_after:.3f} | ECE(late) {ece_late:.3f} -> {ece_T:.3f}")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "k--", lw=.8)
    ax.plot(rel.mean_p, rel.freq, "o-", color=GREY, label="before")
    ax.plot(rel_T.mean_p, rel_T.freq, "o-", color=ACCENT, label=f"after (T = {T:.1f})")
    ax.legend(); ax.set_xlabel("predicted P(late)"); ax.set_ylabel("actual share late"); plt.show()

**✏️ Your answer (Part 4):** Rewrite claim 3 using frequency framing ("of 10 orders the model calls…"). Then say what is wrong with claim 4.

*Your answer:*

---
## Part 5 · One model, three cities (claims 5 and 7)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for a, c in zip(ax, ["Bengaluru", "Pune", "Hyderabad"]):
    m = (test.city == c).values
    r = reliability_table(P_T[m, 2] if T else test.p_late.values[m], y_late[m], n_bins=5)
    a.plot([0, 1], [0, 1], "k--", lw=.8); a.errorbar(r.mean_p, r.freq, yerr=1.96 * r.se, fmt="o-", color=BLUE, capsize=3)
    a.set_title(f"{c} (n = {m.sum()})"); a.set_xlabel("predicted P(late)")
ax[0].set_ylabel("actual share late"); plt.tight_layout(); plt.show()

city_summary = test.groupby("city").agg(mean_error=("error", "mean"), mae=("error", lambda e: e.abs().mean()), share_late=("actual_band", lambda b: (b == "Late").mean()),
                                        mean_p_late=("p_late", "mean")).round(2)
display(city_summary)

worst_city = None   # TODO: the city where the model is least reliable

### Epistemic uncertainty: where has the model never been?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
for c, col in [("Bengaluru", GREY), ("Pune", GREY), ("Hyderabad", ACCENT)]:
    m = df.city == c
    ax.scatter(df.distance_km[m], df.ensemble_sd[m], s=6, alpha=.4, color=col, label=c if c != "Pune" else None)
ax.axvspan(8, df.distance_km.max() + .5, color="#eeeeee", zorder=0); ax.text(8.2, ax.get_ylim()[1] * .9, "outside the training range (> 8 km)", fontsize=8)
ax.set_xlabel("distance (km)"); ax.set_ylabel("spread of 5 models (min)"); ax.legend(); plt.show()

hyd_far_share = None     # TODO: share of Hyderabad orders longer than 8 km
other_far_share = None   # TODO: same for Bengaluru + Pune orders
uncertainty_type = None  # TODO: does ensemble_sd mainly measure 'aleatoric' or 'epistemic' uncertainty?

if ready(hyd_far_share, other_far_share, uncertainty_type):
    print(f"> 8 km: Hyderabad {hyd_far_share:.0%} vs others {other_far_share:.0%} | ensemble_sd measures {uncertainty_type} uncertainty")

**✏️ Your answer (Part 5):** (a) Rewrite claim 5. (b) Is claim 7 true? The app shows the same narrow range everywhere: what should it show for a 10 km Hyderabad order, and which kind of uncertainty would **more Hyderabad data** reduce?

*Your answer:*

---
## Part 6 · Confusion matrix and performance summary (claim 6)

In [ ]:
test["pred_band"] = np.array(BANDS)[test[PCOLS].values.argmax(1)]
cm = pd.crosstab(test.actual_band, test.pred_band).reindex(index=BANDS, columns=BANDS, fill_value=0)
display(cm)

row_norm = None    # TODO: cm divided by its row sums (each row sums to 1): diagonal = recall
precision = None   # TODO: per-band precision as a Series (diagonal / column sums)
recall = None      # TODO: per-band recall as a Series (diagonal / row sums)
accuracy = None    # TODO: overall accuracy
always_ontime = None  # TODO: accuracy of always predicting 'On-time'

if ready(row_norm, precision, recall, accuracy, always_ontime):
    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.imshow(row_norm.values, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3): ax.text(j, i, f"{row_norm.values[i, j]:.0%}\n({cm.values[i, j]})", ha="center", va="center", color="w" if row_norm.values[i, j] > .6 else "k")
    ax.set_xticks(range(3)); ax.set_xticklabels(BANDS); ax.set_yticks(range(3)); ax.set_yticklabels(BANDS); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
    ax.set_title("Row-normalised confusion (counts)"); plt.show()
    summary = pd.DataFrame({"precision": precision, "recall": recall, "support": cm.sum(axis=1)}).round(2)
    display(summary)
    print(f"accuracy {accuracy:.3f} vs always 'On-time' {always_ontime:.3f} | macro recall {recall.mean():.3f}")

**✏️ Your answer (Part 6):** Is claim 6 true? Which band does the model miss most often, and why does that matter for the "may be late" SMS?

*Your answer:*

---
## Part 7 · When should the SMS go out? Choosing an operating point

Missing a late order (no warning) costs **3** units of customer goodwill; an unnecessary SMS costs **1**. Use the **calibrated** P(late) on test orders.

In [ ]:
p_use = P_T[:, 2] if T else test.p_late.values
grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
ops = pd.DataFrame({"threshold": grid})
ops["sms_sent"] = [(p_use >= g).sum() for g in grid]
ops["late_warned"] = [((p_use >= g) & (y_late == 1)).sum() for g in grid]
ops["late_missed"] = y_late.sum() - ops["late_warned"]
ops["false_sms"] = ops["sms_sent"] - ops["late_warned"]

ops["cost"] = None        # TODO: 3 * late_missed + 1 * false_sms
best_sms_threshold = None # TODO: threshold with the lowest cost

if ready(best_sms_threshold) and ops["cost"].notna().all():
    per1000 = ops.set_index("threshold").loc[[0.5, best_sms_threshold], ["late_warned", "late_missed", "false_sms"]] * 1000 / len(test)
    display(per1000.round(0).rename(index=lambda t: f"threshold {t}").rename(columns=lambda c: c + " per 1,000 orders"))
    fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(ops.threshold, ops.cost.astype(float), marker="o", color=BLUE); ax.axvline(best_sms_threshold, color=ACCENT)
    ax.set_xlabel("P(late) threshold"); ax.set_ylabel("cost"); plt.show()

---
## Part 8 · Confidence versus correctness: abstention and high-confidence errors

In [ ]:
P_eval = test[PCOLS].values          # the app's raw (uncalibrated) probabilities: what users actually see
conf, correct = P_eval.max(axis=1), P_eval.argmax(1) == y_te
ths = np.round(np.arange(0.40, 1.0, 0.05), 2)
curve = pd.DataFrame({"threshold": ths, "coverage": [(conf >= t).mean() for t in ths], "accuracy_answered": [correct[conf >= t].mean() for t in ths]}).round(3)
display(curve)

high_conf_errors = None   # TODO: number of test orders with confidence >= 0.9 that were WRONG

if ready(high_conf_errors):
    print(f"Confident (>= 0.9) but wrong: {high_conf_errors} orders")
    t2 = test.assign(conf=conf, correct=correct)
    display(t2[(t2.conf >= .9) & ~t2.correct].groupby("city").size().rename("high-confidence errors by city"))

**✏️ Your answer (Part 8):** Suggest an abstention rule for the speed-band label in the app (e.g. "show no band below confidence X"). What coverage and accuracy does your rule give?

*Your answer:*

---
## Part 9 · Communicating the ETA to a customer

One order: Hyderabad, 9.5 km, ETA from the model. Draw a **quantile dotplot** (20 dots, each 5% of the chance) of plausible arrival times, using the order's own spread: noise **plus** model spread.

In [ ]:
order = df[(df.city == "Hyderabad") & (df.distance_km.between(9, 10))].iloc[0]
spread = np.sqrt((resid_sd if resid_sd else 5) ** 2 + order.ensemble_sd ** 2)
dots = stats.norm.ppf((np.arange(20) + .5) / 20, order.pred_minutes, spread).round()

lo80, hi80 = None, None      # TODO: the 10th and 90th percentiles of the predictive distribution (hint: stats.norm.ppf([0.1, 0.9], order.pred_minutes, spread))
late_dots = None             # TODO: how many of the 20 dots are above 30 minutes?
message = ""                 # TODO: one plain-language sentence for the customer, using the 80% range and a frequency ("about k in 20 deliveries like this...")

fig, ax = plt.subplots(figsize=(8, 2.6))
for v in np.unique(dots):
    k = (dots == v).sum()
    for j in range(k): ax.scatter(v, j + .5, s=170, color=ACCENT if v > 30 else BLUE)
ax.axvline(30, color="k", ls="--", lw=.8); ax.set_yticks([]); ax.set_xlabel("minutes")
ax.set_title(message if message else "(write the customer message)", loc="left", fontsize=10)
plt.show()

---
## Part 10 · Triage Anjali's slide

For each claim decide: **Accept** (correct as stated), **Correct** (useful, but the wording or number must change) or **Reject** (not supported).

In [ ]:
decisions = {
    1: None,   # average error under 1 minute
    2: None,   # 90% range covers 9 in 10
    3: None,   # 90% likely late -> late 90% of the time
    4: None,   # temperature scaling improves accuracy
    5: None,   # equally good in all cities
    6: None,   # beats 'always On-time'
    7: None,   # long Hyderabad orders: just as sure
}

if all(v in ("Accept", "Correct", "Reject") for v in decisions.values()):
    print(pd.Series(decisions).value_counts().to_string())
else:
    print("⚠ Fill in all seven decisions with 'Accept', 'Correct' or 'Reject'.")

**✏️ Your answer (Part 10):** Design the **one-slide model performance view** for the leadership review: a takeaway title, the key metric with its uncertainty, one chart for calibration, one for errors, and a plain-language note. Describe it (or draw it with matplotlib below).

*Your answer:*

---
## Reflection

Answer in 2–4 sentences each. There are no right-or-wrong answers here; we are looking for honest thinking about *your own* process.

**1. Which number fooled you?** Before Part 2, did the "90% range" look trustworthy to you? What about its presentation (a narrow, neat range) made it convincing?

**2. Calibration vs accuracy.** In your own words, why can a model be accurate but badly calibrated? Which one matters more for the SMS decision, and why?

**3. Customers vs analysts.** You wrote a customer message in Part 9. What did you leave out on purpose, and what must never be left out?

**4. Responsibility.** The model has never seen 10 km Hyderabad orders. Who should decide whether the app shows an ETA for them at all, and what evidence from this notebook would you bring to that meeting?

*Your reflection:*